## **Exercise 03 : Aggregations**

### **03.1 Get the schema of the table test**

In [1]:
import pandas as pd
import sqlite3
# create a connection to the database using the library sqlite3
connection = sqlite3.connect('../../data/checking-logs.sqlite')
pd.read_sql('pragma table_info(test)', connection)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


### **03.2 Get only the first 10 rows of the table test to check what the table looks like**

In [2]:
pd.read_sql('select * from test limit 10', connection)

,uid,labname,first_commit_ts,first_view_ts
0,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
1,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
2,user_30,laba04s,2020-04-18 14:51:37.498399,2020-04-17 22:46:26.785035
3,user_14,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447
4,user_14,laba04s,2020-04-18 22:30:30.247628,2020-04-18 10:53:52.623447
5,user_19,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966
6,user_25,laba04,2020-04-20 19:16:50.673054,2020-05-09 23:54:54.260791
7,user_21,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081
8,user_30,project1,2020-04-22 12:36:24.053518,2020-04-17 22:46:26.785035
9,user_21,laba04s,2020-04-22 20:09:21.857747,2020-04-22 22:40:36.824081


### **03.3 Find among all the users the minimum value of the delta between the first commit of the user and the deadline of the corresponding lab using only one query :**

In [3]:
# do this by joining the table with the table deadlines
# the difference should be displayed in hours
# do not take the lab ’project1’ into account, it has longer deadlines and will be an outlier
# the value should be stored in the dataframe df_min with the corresponding uid
df_min = pd.read_sql('with test_to_join as \
                (select uid, labname, first_commit_ts from test group by uid, labname)\
                \
                select uid, min(strftime("%s", first_commit_ts) - deadlines)/ (60*60) as min  \
                from test_to_join left join deadlines on test_to_join.labname = deadlines.labs where labname like "%laba%"', connection)
df_min

,uid,min
0,user_30,-202


### **03.4 Do the same thing, but for the maximum, using only one query, the dataframe name is df_max**

In [4]:
df_max = pd.read_sql('with test_to_join as \
                (select uid, labname, first_commit_ts from test group by uid, labname)\
                \
                select uid, max(strftime("%s", first_commit_ts) - deadlines)/ (60*60) as max  \
                from test_to_join left join deadlines on test_to_join.labname = deadlines.labs where labname like "%laba%"', connection)
df_max

,uid,max
0,user_25,-2


### **03.5 Do he same thing but for the average, using only one query, this time your dataframe should not include the uid column, and the dataframe name is df_avg**

In [5]:
df_avg = pd.read_sql('with test_to_join as \
                (select uid, labname, first_commit_ts from test group by uid, labname)\
                \
                select avg(strftime("%s", first_commit_ts) - deadlines)/ (60*60) as max  \
                from test_to_join left join deadlines on test_to_join.labname = deadlines.labs where labname like "%laba%"', connection)
df_avg

,max
0,-89.687841


### **03.6 We want to test the hypothesis that the users who visited the newsfeed just a few times have the lower delta between the first commit and the deadline. To do this, you need to calculate the correlation coefficient between the number of pageviews and the difference**

In [6]:
# using only one query, create a table with the columns: uid, avg_diff, pageviews
# uid is the uids that exist in the test
# avg_diff is the average delta between the first commit and the lab deadline per user
# pageviews is the number of Newsfeed visits per user
# do not take the lab ’project1’ into account
# store it to the dataframe views_diff

views_diff = pd.read_sql('\
                with avg_diff as \
                (with test_to_join as \
                (select uid, labname, first_commit_ts from test group by uid, labname)\
                \
                select uid, avg(strftime("%s", first_commit_ts) - deadlines)/ (60*60) as avg_diff  \
                from test_to_join left join deadlines on test_to_join.labname = deadlines.labs \
                where labname like "%laba%" group by uid),\
                \
                newsfeed as (select uid, count(datetime) as pageviews from pageviews group by uid)\
                \
                select uid, avg_diff, pageviews from avg_diff left join newsfeed using(uid)', connection)
views_diff

,uid,avg_diff,pageviews
0,user_1,-65.119778,28
1,user_10,-75.242444,89
2,user_14,-159.568796,143
3,user_17,-62.207667,47
4,user_18,-6.368148,3
5,user_19,-99.440417,16
6,user_21,-96.111181,10
7,user_25,-93.474944,179
8,user_28,-86.793833,149
9,user_3,-105.738222,317


In [7]:
# use the Pandas method corr() to calculate the correlation coefficient between the number of pageviews and the difference
views_diff[['avg_diff', 'pageviews']].corr()

,avg_diff,pageviews
avg_diff,1.000000,-0.279143
pageviews,-0.279143,1.000000


In [8]:
connection.close()